# 허깅페이스 다국어 번역 모델 받아 번역기 만들기
[https://huggingface.co/facebook/m2m100_418M](https://huggingface.co/facebook/m2m100_418M)

In [6]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

text = """Trump postpones strike threat: US President Donald Trump told CNN there are 15 points of agreement between the US and Iran after talks this weekend. He announced he will hold off strikes against Iranian energy sites for five days, after earlier threatening an attack if Tehran did not let the Strait of Hormuz fully reopen. Oil prices dropped following Trump’s statement.

• Iran responds: Iran’s foreign ministry said there was “no dialogue” between Tehran and Washington, according to state affiliated media. Separately, the semi-official Fars News Agency, citing what it described as informed Iranian sources, said plans are being prepared for potential actions targeting Tel Aviv and some regional allies of the US and Israel.

• Growing toll: The number of people reported killed in Iran and Lebanon since the start of the conflict is now in the thousands."""

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# translate Hindi to French
tokenizer.src_lang = "en"
encoded_hi = tokenizer(text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("ko"))
result1 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
# => "La vie est comme une boîte de chocolat."

print(result1)


Loading weights: 100%|████████████████████████████████| 512/512 [00:00<00:00, 7922.74it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


['트럼프는 공격 위협을 연기한다 : 미국 대통령 도널드 트럼프는 CNN에 이란과 미국 사이의 협상 후 15 포인트가 있다고 말했다. 그는 이란 에너지 시설에 대한 공격을 5 일 동안 중단 할 것이라고 발표 한 후 이전에 테헤란이 호르무즈 스트리트를 완전히 재개하지 않으면 공격을 위협했다. 석유 가격은 트럼프의 진술에 따라 떨어졌다. • 이란은 응답 : 이란 외무부는 테헤란과 워싱턴 사이에 "대화가 없었다"고 말했다.']


In [9]:
import gradio as gr
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# =========================
# 1. 모델 로드
# =========================
MODEL_NAME = "facebook/m2m100_418M"

print("모델 로드 중...")
tokenizer = M2M100Tokenizer.from_pretrained(MODEL_NAME)
model = M2M100ForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"사용 장치: {device}")

# =========================
# 2. 언어 목록
# M2M100 언어코드 기준
# =========================
LANGUAGE_MAP = {
    "Korean": "ko",
    "English": "en",
    "Japanese": "ja",
    "Chinese": "zh",
    "French": "fr",
    "German": "de",
    "Spanish": "es",
    "Italian": "it",
    "Russian": "ru",
    "Hindi": "hi",
    "Portuguese": "pt",
    "Arabic": "ar",
}

LANGUAGE_NAMES = list(LANGUAGE_MAP.keys())


# =========================
# 3. 번역 함수
# =========================
def translate_text(text, src_lang_name, tgt_lang_name):
    if text is None or text.strip() == "":
        return ""

    src_lang = LANGUAGE_MAP[src_lang_name]
    tgt_lang = LANGUAGE_MAP[tgt_lang_name]

    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_length=1024
    )

    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return result


# =========================
# 4. 언어 스왑 함수
# =========================
def swap_languages(src_lang, tgt_lang, src_text, tgt_text):
    return tgt_lang, src_lang, tgt_text, src_text


# =========================
# 5. 글자수 표시
# =========================
def update_char_count(text):
    if text is None:
        return "0 / 3000"
    return f"{len(text)} / 3000"


# =========================
# 6. 복사 버튼용 함수
# Gradio에서는 클립보드 직접 제어 제한이 있어
# 텍스트 반환으로 대체하거나 브라우저 기본 복사 활용
# =========================
def passthrough_text(text):
    return text if text else ""


# =========================
# 7. UI
# =========================
custom_css = """
body {
    background: #f7f7f7;
}
.container-wrap {
    max-width: 1600px;
    margin: 0 auto;
}
.main-card {
    background: white;
    border-radius: 18px;
    border: 1px solid #e5e7eb;
    overflow: hidden;
    box-shadow: 0 2px 10px rgba(0,0,0,0.04);
}
.panel-title {
    font-size: 18px;
    font-weight: 600;
    color: #333;
}
.char-count {
    text-align: right;
    color: #777;
    font-size: 14px;
    margin-top: -8px;
}
.translate-btn button {
    background: #22c55e !important;
    color: white !important;
    border: none !important;
    font-size: 20px !important;
    font-weight: 700 !important;
    min-height: 80px !important;
    border-radius: 0 !important;
}
.swap-btn button {
    min-width: 52px !important;
    min-height: 52px !important;
    border-radius: 999px !important;
    font-size: 20px !important;
}
.footer-btn button {
    min-height: 48px !important;
    font-size: 16px !important;
}
textarea {
    font-size: 20px !important;
    line-height: 1.6 !important;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.Markdown("## M2M100 번역기")

    with gr.Row(elem_classes=["container-wrap"]):
        with gr.Column(scale=1):
            with gr.Group(elem_classes=["main-card"]):
                with gr.Row():
                    src_lang = gr.Dropdown(
                        choices=LANGUAGE_NAMES,
                        value="Korean",
                        label="",
                        interactive=True
                    )

                    swap_btn = gr.Button("⇄", elem_classes=["swap-btn"])

                    tgt_lang = gr.Dropdown(
                        choices=LANGUAGE_NAMES,
                        value="English",
                        label="",
                        interactive=True
                    )

                with gr.Row():
                    src_text = gr.Textbox(
                        lines=14,
                        max_lines=14,
                        placeholder="Type the text to translate",
                        show_label=False
                    )

                    tgt_text = gr.Textbox(
                        lines=14,
                        max_lines=14,
                        placeholder="Translation result",
                        show_label=False,
                        interactive=False
                    )

                with gr.Row():
                    char_count = gr.Markdown("0 / 3000")

                with gr.Row():
                    copy_src_btn = gr.Button("원문 복사용", elem_classes=["footer-btn"])
                    copy_tgt_btn = gr.Button("번역문 복사용", elem_classes=["footer-btn"])
                    translate_btn = gr.Button("Translate", elem_classes=["translate-btn"])

        copied_src = gr.Textbox(label="원문 복사용 출력", visible=False)
        copied_tgt = gr.Textbox(label="번역문 복사용 출력", visible=False)

    # 입력할 때 글자 수 업데이트
    src_text.change(
        fn=update_char_count,
        inputs=src_text,
        outputs=char_count
    )

    # 번역 버튼
    translate_btn.click(
        fn=translate_text,
        inputs=[src_text, src_lang, tgt_lang],
        outputs=tgt_text
    )

    # 엔터 입력 시도 바로 번역되게 하려면 submit 사용
    src_text.submit(
        fn=translate_text,
        inputs=[src_text, src_lang, tgt_lang],
        outputs=tgt_text
    )

    # 스왑 버튼
    swap_btn.click(
        fn=swap_languages,
        inputs=[src_lang, tgt_lang, src_text, tgt_text],
        outputs=[src_lang, tgt_lang, src_text, tgt_text]
    )

    # 복사용 출력
    copy_src_btn.click(
        fn=passthrough_text,
        inputs=src_text,
        outputs=copied_src
    )

    copy_tgt_btn.click(
        fn=passthrough_text,
        inputs=tgt_text,
        outputs=copied_tgt
    )

demo.launch(inline=False,  share=False)

모델 로드 중...


Loading weights: 100%|████████████████████████████████| 512/512 [00:00<00:00, 6648.39it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


사용 장치: cuda


/tmp/ipykernel_5646/3965062860.py:142: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
demo.close()

Closing server running on port: 7860


# 한영-영한 번역 결과를 음성으로 읽어주는 서비스

In [ ]:
# !pip install "transformers>=4.33" accelerate soundfile
# !pip install "kokoro>=0.9.2"
# !pip install uroman
# !sudo apt-get -y install espeak-ng

In [10]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np

from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    VitsModel,
    AutoTokenizer,
)
from kokoro import KPipeline   # Kokoro-82M 파이썬 래퍼
from uroman import Uroman      # 🔥 pip install uroman

# -----------------------------
# 0. 디바이스 (GPU 우선)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 1. 번역 모델 (M2M100)
# -----------------------------
translation_model_name = "facebook/m2m100_418M"
translation_model = M2M100ForConditionalGeneration.from_pretrained(
    translation_model_name
).to(device)
translation_tokenizer = M2M100Tokenizer.from_pretrained(translation_model_name)

# -----------------------------
# 2. 영어 TTS: Kokoro-82M
# -----------------------------
kokoro_pipeline = KPipeline(lang_code="a")

def tts_english_kokoro(text: str, path: str = "tts_en.wav") -> str | None:
    text = (text or "").strip()
    if not text:
        return None

    generator = kokoro_pipeline(text, voice="af_heart")

    chunks = []
    for _, _, audio in generator:
        chunks.append(audio)

    if not chunks:
        return None

    audio_cat = np.concatenate(chunks)
    sf.write(path, audio_cat, 24000)
    return path

# -----------------------------
# 3. 한국어 TTS: MMS-TTS-KOR (VITS)
#    🔥 pip uroman 사용
# -----------------------------
kor_tts_name = "facebook/mms-tts-kor"
kor_tts_model = VitsModel.from_pretrained(kor_tts_name).to(device)
kor_tts_tokenizer = AutoTokenizer.from_pretrained(kor_tts_name)

uroman = Uroman()   # 🔥 한 번만 초기화하면 됨

def tts_korean_mms(text: str, path: str = "tts_ko.wav") -> str | None:
    text = (text or "").strip()
    if not text:
        return None

    # 1) 🔥 uroman 로마자 변환 (찾는 발음형)
    roman_text = uroman.romanize_string(text)

    # 2) 토크나이저 입력
    inputs = kor_tts_tokenizer(roman_text, return_tensors="pt").to(device)

    # 3) 모델 인퍼런스
    with torch.no_grad():
        outputs = kor_tts_model(**inputs).waveform

    wav = outputs.squeeze().cpu().numpy()
    if wav.size == 0:
        return None

    sf.write(path, wav, samplerate=kor_tts_model.config.sampling_rate)
    return path

# -----------------------------
# 4. 언어 매핑
# -----------------------------
LANGUAGES = {
    "Korean": "ko",
    "English": "en",
    "Chinese": "zh",
    "Japanese": "ja",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Hindi": "hi",
}

# -----------------------------
# 5. 번역 + TTS 함수
# -----------------------------
def translate_and_speak(text, src_lang_name, tgt_lang_name):
    text = text or ""
    if not text.strip():
        return "", None

    src_lang = LANGUAGES[src_lang_name]
    tgt_lang = LANGUAGES[tgt_lang_name]

    # ----- 번역 -----
    if src_lang == tgt_lang:
        translated_text = text.strip()
    else:
        translation_tokenizer.src_lang = src_lang
        encoded = translation_tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            generated_tokens = translation_model.generate(
                **encoded,
                forced_bos_token_id=translation_tokenizer.get_lang_id(tgt_lang),
                max_length=512,
            )

        translated_text = translation_tokenizer.batch_decode(
            generated_tokens, skip_special_tokens=True
        )[0]

    # ----- TTS -----
    audio_path = None
    if tgt_lang == "en":
        audio_path = tts_english_kokoro(translated_text)
    elif tgt_lang == "ko":
        audio_path = tts_korean_mms(translated_text)

    return translated_text, audio_path

# -----------------------------
# 6. Gradio UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 다국어 번역 + TTS (영어 Kokoro + 한국어 MMS)")

    with gr.Row():
        src_lang = gr.Dropdown(list(LANGUAGES.keys()), label="원본 언어", value="Korean")
        tgt_lang = gr.Dropdown(list(LANGUAGES.keys()), label="번역 언어", value="English")

    with gr.Row():
        input_text = gr.Textbox(lines=8, label="입력 텍스트")
        output_text = gr.Textbox(lines=8, label="번역 결과", interactive=False)

    audio_output = gr.Audio(label="음성 출력", type="filepath")

    translate_button = gr.Button("번역 + 음성 생성")
    translate_button.click(
        fn=translate_and_speak,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=[output_text, audio_output],
    )

    demo.queue()

demo.launch(inline=False, share=False)


Loading weights: 100%|████████████████████████████████| 512/512 [00:00<00:00, 7586.57it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


/home/haram/miniforge3/envs/hug/lib/python3.11/site-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/home/haram/miniforge3/envs/hug/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 22.5 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Loading weights: 100%|████████████████████████████████| 762/762 [00:00<00:00, 1095.68it/s]


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
